<a href="https://colab.research.google.com/github/meetparmar392005/AI-Based-Consumer-Complaint-Classification-Priority-Prediction-System/blob/main/mini_project_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load the CSV file into a pandas DataFrame
df = pd.read_csv('/content/cleaned_cc_50k.csv')

# Display the first 5 rows of the DataFrame
df.head()

,complaint_text,category,priority
0,Unauthorized transaction of ₹12355 detected in...,Banking,Critical
1,Credit card payment of ₹12453 is not reflected...,Banking,Critical
2,Incorrect mobile bill of ₹2232 generated this ...,Telecom,Medium
3,Voltage fluctuations damaging electrical appli...,Electricity,Medium
4,Street lights not functioning in my locality,Electricity,Critical


This cell prints the shape of the DataFrame before and after removing duplicate rows, then removes the duplicate rows in-place using `df.drop_duplicates(inplace=True)`.

In [ ]:
print(f"Shape of DataFrame before removing duplicates: {df.shape}")
df.drop_duplicates(inplace=True)
print(f"Shape of DataFrame after removing duplicates: {df.shape}")

Shape of DataFrame before removing duplicates: (50000, 3)
Shape of DataFrame after removing duplicates: (9466, 3)


This cell imports the `pandas` library, loads the `cleaned_cc_50k.csv` file into a DataFrame named `df`, and then displays the first 5 rows of the DataFrame to provide a quick overview of its content.

In [ ]:
print(df.shape)
print(df.columns)
print(df['category'].value_counts())


(9466, 3)
Index(['complaint_text', 'category', 'priority'], dtype='object')
category
Banking        4663
Electricity    1615
Telecom        1595
Water          1593
Name: count, dtype: int64


This cell prints the shape of the DataFrame, lists the column names, and then shows the distribution of values in the 'category' column using `value_counts()`.

In [ ]:
df

,complaint_text,category,priority
0,Unauthorized transaction of ₹12355 detected in...,Banking,Critical
1,Credit card payment of ₹12453 is not reflected...,Banking,Critical
2,Incorrect mobile bill of ₹2232 generated this ...,Telecom,Medium
3,Voltage fluctuations damaging electrical appli...,Electricity,Medium
4,Street lights not functioning in my locality,Electricity,Critical
...,...,...,...
49983,ATM deducted ₹35149 but cash was not dispensed...,Banking,High
49986,Electricity bill amount of ₹31662 is incorrect,Electricity,Medium
49987,ATM deducted ₹22743 but cash was not dispensed...,Banking,High
49991,Credit card payment of ₹8031 is not reflected ...,Banking,Critical


This cell simply displays the entire DataFrame `df` after the previous operations, showing its current state including the cleaned data and remaining columns.

In [ ]:
df = df[['complaint_text', 'category']]


This cell reassigns the DataFrame `df` to only include the 'complaint_text' and 'category' columns, effectively dropping the 'priority' column. This prepares the data for text classification where only the complaint text and its category are relevant.

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['complaint_text'] = df['complaint_text'].apply(clean_text)


/tmp/ipython-input-3236313480.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['complaint_text'] = df['complaint_text'].apply(clean_text)


This cell defines a `clean_text` function that converts text to lowercase, removes non-alphabetic characters, and standardizes whitespace. It then applies this function to the 'complaint_text' column of the DataFrame to clean the text data.

In [ ]:
df

,complaint_text,category
0,unauthorized transaction of detected in my acc...,Banking
1,credit card payment of is not reflected in my ...,Banking
2,incorrect mobile bill of generated this month,Telecom
3,voltage fluctuations damaging electrical appli...,Electricity
4,street lights not functioning in my locality,Electricity
...,...,...
49983,atm deducted but cash was not dispensed at surat,Banking
49986,electricity bill amount of is incorrect,Electricity
49987,atm deducted but cash was not dispensed at surat,Banking
49991,credit card payment of is not reflected in my ...,Banking


This cell displays the DataFrame `df` after the text cleaning process, showing the updated 'complaint_text' column with cleaned text.

In [ ]:
X = df["complaint_text"]
y = df["category"]
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (7572,)
Test size: (1894,)


This cell separates the features (complaint text) into `X` and the target labels (category) into `y`. It then uses `train_test_split` from `sklearn.model_selection` to divide the data into training and testing sets, ensuring a stratified split based on the `y` variable. Finally, it prints the shapes of the resulting training and testing sets.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


This cell initializes a `TfidfVectorizer` to convert text data into numerical feature vectors using the TF-IDF (Term Frequency-Inverse Document Frequency) method. It fits the vectorizer on the training data (`X_train`) and then transforms both the training and testing data into TF-IDF vectors.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)


LogisticRegression(max_iter=1000)

This cell initializes a Logistic Regression model with `max_iter=1000` to ensure convergence. It then trains the model using the TF-IDF vectorized training data (`X_train_vec`) and the corresponding training labels (`y_train`).

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))


              precision    recall  f1-score   support

     Banking       1.00      1.00      1.00       933
 Electricity       1.00      0.99      1.00       323
     Telecom       1.00      1.00      1.00       319
       Water       1.00      1.00      1.00       319

    accuracy                           1.00      1894
   macro avg       1.00      1.00      1.00      1894
weighted avg       1.00      1.00      1.00      1894

Accuracy: 0.9989440337909187


This cell evaluates the performance of the trained Logistic Regression model. It predicts the categories for the test set (`X_test_vec`) and then prints a detailed classification report (including precision, recall, f1-score, and support for each class) and the overall accuracy score.

In [ ]:
import pickle

pickle.dump(model, open("complaint_classifier.pkl", "wb"))
pickle.dump(vectorizer, open("tfidf_vectorizer.pkl", "wb"))


testing

This cell uses the `pickle` library to save the trained Logistic Regression model (`model`) and the `TfidfVectorizer` (`vectorizer`) to disk. This allows the trained model and vectorizer to be reloaded and used later without retraining.

In [ ]:
def predict_complaint(text):
    vec = vectorizer.transform([text])
    pred = model.predict(vec)
    return pred[0]

# Test cases
samples = [
    "money removed from my account but no cash received",
    "my monthly bill amount seems higher than usual",
    "there is no water supply in my area since morning",
    "internet is not working even after recharge",
    "lights are flickering continuously at home"
]

for s in samples:
    print("Complaint:", s)
    print("Predicted:", predict_complaint(s))
    print("-" * 50)

Complaint: money removed from my account but no cash received
Predicted: Banking
--------------------------------------------------
Complaint: my monthly bill amount seems higher than usual
Predicted: Banking
--------------------------------------------------
Complaint: there is no water supply in my area since morning
Predicted: Water
--------------------------------------------------
Complaint: internet is not working even after recharge
Predicted: Telecom
--------------------------------------------------
Complaint: lights are flickering continuously at home
Predicted: Banking
--------------------------------------------------


This cell defines a `predict_complaint` function that takes raw text, vectorizes it using the previously fitted `vectorizer`, and then predicts the complaint category using the trained `model`. It then demonstrates this function with a few sample complaint texts.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Create pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])

# Train pipeline
pipeline.fit(X_train, y_train)

print("Pipeline trained successfully!")


Pipeline trained successfully!


This cell creates a `Pipeline` that sequentially applies `TfidfVectorizer` and `LogisticRegression`. This simplifies the process of training and prediction by encapsulating both steps. The pipeline is then trained on the original (unvectorized) training data `X_train` and `y_train`.

In [ ]:
import joblib

joblib.dump(pipeline, "model.pkl")


['model.pkl']

This cell uses `joblib.dump` to save the trained `pipeline` object to a file named 'model.pkl'. This is a convenient way to persist the entire pipeline, including both the vectorizer and the classifier, for later use.

In [ ]:
import joblib

# Load pipeline
model = joblib.load("model.pkl")

# Test
sample_text = ["internet not working after recharge"]

prediction = model.predict(sample_text)

print("Complaint:", sample_text[0])
print("Predicted Category:", prediction[0])


Complaint: internet not working after recharge
Predicted Category: Telecom


This cell demonstrates how to load the saved pipeline using `joblib.load`. It then uses the loaded `model` to make a prediction on a new sample text, showcasing the end-to-end prediction process with the pipeline.

In [ ]:
hard_samples = [
    "My payment was successful but the service is still not active",
    "Amount deducted but connection not restored",
    "I paid the bill but the issue still persists",
    "Service was disconnected without any prior notice",
    "Unexpected charges added to my account",
    "Transaction completed but nothing changed on my end",
    "Monthly charges seem incorrect",
    "The system shows payment received but service not resumed",
    "My complaint was closed but the issue is still unresolved",
    "Frequent interruptions affecting daily activities",
    "Not working since morning",
    "Problem still continues",
    "Charges are too high",
    "Deduction happened twice",
    "No supply today",
    "Recharge successful but amount deducted twice",
    "Bill paid but connection still inactive",
    "Account shows negative balance after last payment",
    "Network issue after bill payment",
    "Meter reading seems incorrect"
]

for text in hard_samples:
    print("Complaint:", text)
    print("Prediction:", model.predict([text])[0])
    print("-" * 60)


Complaint: My payment was successful but the service is still not active
Prediction: Banking
------------------------------------------------------------
Complaint: Amount deducted but connection not restored
Prediction: Banking
------------------------------------------------------------
Complaint: I paid the bill but the issue still persists
Prediction: Telecom
------------------------------------------------------------
Complaint: Service was disconnected without any prior notice
Prediction: Banking
------------------------------------------------------------
Complaint: Unexpected charges added to my account
Prediction: Banking
------------------------------------------------------------
Complaint: Transaction completed but nothing changed on my end
Prediction: Banking
------------------------------------------------------------
Complaint: Monthly charges seem incorrect
Prediction: Banking
------------------------------------------------------------
Complaint: The system shows payme

This cell tests the loaded pipeline with a list of 'hard samples' which are more ambiguous complaint texts. It iterates through each sample, predicts its category using the `model`, and prints the complaint and its predicted category. This helps in understanding the model's performance on challenging inputs.